# Real-Time Fraud Prediction Publisher

This notebook consumes transactions from the Kafka topic:

fraud-transactions

predicts fraud using the trained Random Forest model

and publishes enriched prediction results to another Kafka topic:

fraud-predictions

This notebook acts as the real-time Machine Learning Prediction Engine.

## Imports

In [ ]:
import json
import joblib
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Create Spark Session

In [ ]:
spark = (
    SparkSession.builder
    .appName("FraudPredictionPublisher")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

## Stop Existing Streams

In [ ]:
for stream in spark.streams.active:
    stream.stop()

print("Existing streams stopped.")

## Load Best Model

In [ ]:
model = joblib.load("../models/best_model.pkl")

print(model)

## Load Metadata

In [ ]:
with open("../models/best_model_metadata.json","r") as f:
    metadata=json.load(f)

metadata

## Read Kafka Stream

In [ ]:
stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers","localhost:9092")
    .option("subscribe","fraud-transactions")
    .option("startingOffsets","latest")
    .load()
)

## Schema

In [ ]:
schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("event_timestamp", StringType()),
    StructField("Time", DoubleType()),

    *[
        StructField(f"V{i}", DoubleType())
        for i in range(1,29)
    ],

    StructField("Amount", DoubleType()),
    StructField("Class", DoubleType())
])

## Parse JSON

In [ ]:
parsed_df = (
    stream_df
    .selectExpr("CAST(value AS STRING)")
    .select(
        from_json(
            col("value"),
            schema
        ).alias("data")
    )
    .select("data.*")
)

## Verify Schema

In [ ]:
parsed_df.printSchema()

## Feature Engineering

In [ ]:
feature_df = (
    parsed_df

    .withColumn(
        "Large_Transaction",
        when(col("Amount") > 200, 1).otherwise(0)
    )

    .withColumn(
        "Log_Amount",
        log1p(col("Amount"))
    )

    .withColumn(
        "Amount_Level",
        when(col("Amount") < 10, 0)
        .when(col("Amount") < 100, 1)
        .when(col("Amount") < 500, 2)
        .otherwise(3)
    )
)

## Prediction Function

In [ ]:
from pyspark.sql import Row

def predict_batch(batch_df, batch_id):

    pdf = batch_df.toPandas()

    if pdf.empty:
        return spark.createDataFrame([], schema="""
            transaction_id STRING,
            event_timestamp STRING,
            Amount DOUBLE,
            Prediction INT,
            Fraud_Probability DOUBLE,
            Status STRING,
            Risk_Level STRING
        """)

    X = pdf[model.feature_names_in_]

    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:,1]

    pdf["Prediction"] = predictions
    pdf["Fraud_Probability"] = probabilities.round(6)

    pdf["Status"] = pdf["Prediction"].map({
        0:"Genuine",
        1:"Fraud"
    })

    def risk(prob):

        if prob>=0.90:
            return "Critical"

        elif prob>=0.70:
            return "High"

        elif prob>=0.40:
            return "Medium"

        else:
            return "Low"

    pdf["Risk_Level"]=pdf["Fraud_Probability"].apply(risk)

    rows=[]

    for _,row in pdf.iterrows():

        rows.append(Row(

            transaction_id=row["transaction_id"],

            event_timestamp=row["event_timestamp"],

            Amount=float(row["Amount"]),

            Prediction=int(row["Prediction"]),

            Fraud_Probability=float(row["Fraud_Probability"]),

            Status=row["Status"],

            Risk_Level=row["Risk_Level"]

        ))

    return spark.createDataFrame(rows)

## Output Schema

In [ ]:
prediction_schema = StructType([

    StructField("transaction_id", StringType()),

    StructField("event_timestamp", StringType()),

    StructField("Amount", DoubleType()),

    StructField("Prediction", IntegerType()),

    StructField("Fraud_Probability", DoubleType()),

    StructField("Status", StringType()),

    StructField("Risk_Level", StringType())

])

## Apply Prediction

In [ ]:
prediction_stream = (

    feature_df.writeStream

    .foreachBatch(
        lambda batch_df, batch_id:
        predict_batch(batch_df, batch_id)
    )

)

## Convert Predictions to JSON

In [ ]:
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda x: json.dumps(x).encode("utf-8")
)


def predict_batch(batch_df, batch_id):

    pdf = batch_df.toPandas()

    if pdf.empty:
        return

    # -------------------------
    # Arrange features exactly
    # -------------------------

    X = pdf[model.feature_names_in_]

    predictions = model.predict(X)

    probabilities = model.predict_proba(X)[:, 1]

    pdf["Prediction"] = predictions

    pdf["Fraud_Probability"] = probabilities.round(6)

    pdf["Status"] = pdf["Prediction"].map({
        0: "Genuine",
        1: "Fraud"
    })

    def risk(prob):

        if prob >= 0.90:
            return "Critical"

        elif prob >= 0.70:
            return "High"

        elif prob >= 0.40:
            return "Medium"

        else:
            return "Low"

    pdf["Risk_Level"] = pdf["Fraud_Probability"].apply(risk)

    print("\n")
    print("=" * 120)
    print(f"Publishing Batch {batch_id}")
    print("=" * 120)

    for _, row in pdf.iterrows():

        message = {

            "transaction_id": row["transaction_id"],

            "event_timestamp": row["event_timestamp"],

            "Amount": float(row["Amount"]),

            "Prediction": int(row["Prediction"]),

            "Fraud_Probability": float(row["Fraud_Probability"]),

            "Risk_Level": row["Risk_Level"],

            "Status": row["Status"]

        }

        producer.send(
            "fraud-predictions",
            value=message
        )

        print(message)

    producer.flush()

    print(f"\nPublished {len(pdf)} prediction(s) to Kafka.")

## Start Prediction Publisher

In [ ]:
query = (

    feature_df

    .writeStream

    .foreachBatch(predict_batch)

    .outputMode("append")

    .option(
        "checkpointLocation",
        "../checkpoints/prediction_publisher"
    )

    .start()

)

## Keep Alive

In [ ]:
query.awaitTermination() 